In [7]:
COCO_INSTANCE_CATEGORY_NAMES = [
    '__background__', 'person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus',
    'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'N/A', 'stop sign',
    'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow',
    'elephant', 'bear', 'zebra', 'giraffe', 'N/A', 'backpack', 'umbrella', 'N/A', 'N/A',
    'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard', 'sports ball',
    'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 'tennis racket',
    'bottle', 'N/A', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl',
    'banana', 'apple', 'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza',
    'donut', 'cake', 'chair', 'couch', 'potted plant', 'bed', 'N/A', 'dining table',
    'N/A', 'N/A', 'toilet', 'N/A', 'tv', 'laptop', 'mouse', 'remote', 'keyboard', 'cell phone',
    'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'N/A', 'book',
    'clock', 'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush'
]

Utils

In [2]:
import cv2
import numpy as np
import random
import torch
#from coco_names import COCO_INSTANCE_CATEGORY_NAMES as coco_names
coco_names=COCO_INSTANCE_CATEGORY_NAMES
COLORS = np.random.uniform(0, 255, size=(len(coco_names), 3))


def get_outputs(image, model, threshold):
    with torch.no_grad():
        # forward pass of the image through the modle
        outputs = model(image)
    
    # get all the scores
    scores = list(outputs[0]['scores'].detach().cpu().numpy())
    # index of those scores which are above a certain threshold
    thresholded_preds_inidices = [scores.index(i) for i in scores if i > threshold]
    thresholded_preds_count = len(thresholded_preds_inidices)
    # get the masks
    masks = (outputs[0]['masks']>0.5).squeeze().detach().cpu().numpy()
    # discard masks for objects which are below threshold
    masks = masks[:thresholded_preds_count]
    # get the bounding boxes, in (x1, y1), (x2, y2) format
    boxes = [[(int(i[0]), int(i[1])), (int(i[2]), int(i[3]))]  for i in outputs[0]['boxes'].detach().cpu()]
    # discard bounding boxes below threshold value
    boxes = boxes[:thresholded_preds_count]
    # get the classes labels
    labels = [coco_names[i] for i in outputs[0]['labels']]
    return masks, boxes, labels

In [1]:
def draw_segmentation_map(image, masks, boxes, labels):
    alpha = 1 
    beta = 0.6 # transparency for the segmentation map
    gamma = 0 # scalar added to each sum
    for i in range(len(masks)):
        red_map = np.zeros_like(masks[i]).astype(np.uint8)
        green_map = np.zeros_like(masks[i]).astype(np.uint8)
        blue_map = np.zeros_like(masks[i]).astype(np.uint8)
        # apply a randon color mask to each object
        color = COLORS[random.randrange(0, len(COLORS))]
        red_map[masks[i] == 1], green_map[masks[i] == 1], blue_map[masks[i] == 1]  = color
        # combine all the masks into a single image
        segmentation_map = np.stack([red_map, green_map, blue_map], axis=2)
        #convert the original PIL image into NumPy format
        image = np.array(image)
        # convert from RGN to OpenCV BGR format
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
        # apply mask on the image
        cv2.addWeighted(image, alpha, segmentation_map, beta, gamma, image)
        # draw the bounding boxes around the objects
        cv2.rectangle(image, boxes[i][0], boxes[i][1], color=color, 
                      thickness=2)
        # put the label text above the objects
        cv2.putText(image , labels[i], (boxes[i][0][0], boxes[i][0][1]-10), 
                    cv2.FONT_HERSHEY_SIMPLEX, 1, color, 
                    thickness=2, lineType=cv2.LINE_AA)
    
    return image

mask rcnn

In [1]:
import os
import sys

sys.path.append(os.path.join(os.getcwd(), *tuple(['..'])))
from features import transforms as T
from features import build_features
#train_dataset = build_features.LBD_Dataset(dataset_train_location, T.Compose([T.ToTensor()]))

In [2]:
dataset_train_location = '/home/mahirwar/Desktop/Monika/npsad_data/monika/LBD/image_dataset/train'
dataset_test_location = '/home/mahirwar/Desktop/Monika/npsad_data/monika/LBD/image_dataset/test'
train_dataset = build_features.LBD_Dataset(dataset_train_location, T.Compose([T.ToTensor()]))

In [3]:
train_dataset[0]

(tensor([[[0.7961, 0.8353, 0.8941,  ..., 0.9333, 0.9412, 0.9608],
          [0.8275, 0.8667, 0.9020,  ..., 0.9569, 0.9922, 0.9804],
          [0.8627, 0.8863, 0.9137,  ..., 0.9216, 0.9255, 0.9294],
          ...,
          [0.9255, 0.9333, 0.9451,  ..., 0.8039, 0.8235, 0.8235],
          [0.9373, 0.9412, 0.9529,  ..., 0.8196, 0.8196, 0.8157],
          [0.9529, 0.9529, 0.9569,  ..., 0.8510, 0.8510, 0.8588]],
 
         [[0.7412, 0.7804, 0.8549,  ..., 0.9373, 0.9333, 0.9529],
          [0.7725, 0.8118, 0.8706,  ..., 0.9412, 0.9922, 0.9804],
          [0.8118, 0.8353, 0.8745,  ..., 0.9294, 0.9216, 0.9255],
          ...,
          [0.9176, 0.9255, 0.9451,  ..., 0.7843, 0.7961, 0.7961],
          [0.9333, 0.9373, 0.9529,  ..., 0.7922, 0.7922, 0.7882],
          [0.9451, 0.9451, 0.9490,  ..., 0.8118, 0.8314, 0.8392]],
 
         [[0.6980, 0.7373, 0.8078,  ..., 0.9137, 0.9451, 0.9647],
          [0.7294, 0.7686, 0.8275,  ..., 1.0000, 1.0000, 0.9882],
          [0.7765, 0.8000, 0.8392,  ...,

In [4]:
from model_mrcnn import _default_mrcnn_config, build_default
train_config = dict(
    epochs = 50,
    batch_size = 6,
    num_classes = 3,
    device_id = 0,
    ckpt_freq =500,
    eval_freq = 25,
)

test_config = dict(
    batch_size = 1
)

model_config = _default_mrcnn_config(num_classes=1+train_config['num_classes']).config


from model_mrcnn import _default_mrcnn_config, build_default
model = build_default(model_config, im_size=1024)

/home/mahirwar/miniconda3/envs/kfold_amy_plaque1/lib/python3.9/site-packages/torchvision/models/_utils.py:135: UserWarning: Using 'backbone_name' as positional parameter(s) is deprecated since 0.13 and may be removed in the future. Please use keyword parameter(s) instead.
  warnings.warn(
/home/mahirwar/miniconda3/envs/kfold_amy_plaque1/lib/python3.9/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/mahirwar/miniconda3/envs/kfold_amy_plaque1/lib/python3.9/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [5]:
model

GeneralizedRCNN(
  (transform): GeneralizedRCNNTransform(
      Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
      Resize(min_size=(1024,), max_size=1024, mode='bilinear')
  )
  (backbone): BackboneWithFPN(
    (body): IntermediateLayerGetter(
      (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): FrozenBatchNorm2d(64, eps=1e-05)
      (relu): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0): Bottleneck(
          (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn1): FrozenBatchNorm2d(64, eps=1e-05)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): FrozenBatchNorm2d(64, eps=1e-05)
          (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn3): FrozenBatchNorm2d(256, eps=1e-05)
         

In [6]:
import torch
if torch.cuda.is_available():
    assert train_config['device_id'] >= 0 and train_config['device_id'] < torch.cuda.device_count()
    device = torch.device('cuda', train_config['device_id'])
model = model.to(device)
model.train(True)

/home/mahirwar/miniconda3/envs/kfold_amy_plaque1/lib/python3.9/site-packages/torch/cuda/__init__.py:546: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


GeneralizedRCNN(
  (transform): GeneralizedRCNNTransform(
      Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
      Resize(min_size=(1024,), max_size=1024, mode='bilinear')
  )
  (backbone): BackboneWithFPN(
    (body): IntermediateLayerGetter(
      (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): FrozenBatchNorm2d(64, eps=1e-05)
      (relu): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0): Bottleneck(
          (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn1): FrozenBatchNorm2d(64, eps=1e-05)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): FrozenBatchNorm2d(64, eps=1e-05)
          (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn3): FrozenBatchNorm2d(256, eps=1e-05)
         

In [7]:

from typing import Callable, Dict, List, Optional, Set
from collections import OrderedDict


def get_loss_fn(weights, default=0.):
    
    def compute_loss_fn(losses):
        item = lambda k: (k, losses[k].item())
        metrics = OrderedDict(list(map(item, [k for k in weights.keys() if k in losses.keys()] + [k for k in losses.keys() if k not in weights.keys()])))

        loss = sum(map(lambda k: losses[k] * (weights[k] if weights is not None and k in weights.keys() else default), losses.keys()))
        return loss, metrics
    return compute_loss_fn


optim_config = dict(
    # cls=grad_optim.GradSGD,
    cls=torch.optim.SGD,
    defaults=dict(lr=1. * (10. ** (-2)))  #-4 is too slow 
)

loss_names = 'objectness rpn_box_reg classifier box_reg mask'.split()
loss_weights = [1., 4., 1., 4., 1.,]
loss_weights = OrderedDict([(f'loss_{name}', weight) for name, weight in zip(loss_names, loss_weights)])

loss_fn = get_loss_fn(loss_weights)

optimizer = optim_config['cls']([dict(params=list(model.parameters()))], **optim_config['defaults'])

In [8]:

from torch import nn, Tensor
import torch.optim

def train_one_epoch(
    model: torch.nn.Module,
    loss_fn: Callable[[Dict[str, Tensor]], Tensor],
    optimizer: torch.optim.Optimizer,
    data_loader: torch.utils.data.DataLoader,
    device: torch.device,
    epoch: int = 1,
    log_freq: int = 10,) -> None:

    assert model.training
    model_params = set(model.parameters())
    model_devices = set([p.device for p in model_params])
    assert model_devices == set([device]) # validate model params device
    for g in optimizer.param_groups: # validate optimizer params
        assert set(g['params']).issubset(model_params)

    log_metrics = list()

    for i, (images, targets) in enumerate(train_data_loader):
        images = [image.to(device) for image in images]
        targets = [dict([(k, v.to(device)) for k, v in target.items()]) for target in targets]
        # visualize_augmentations(images , targets)
        # pdb.set_trace()
        optimizer.zero_grad()
        loss, metrics = loss_fn(model.forward(images, targets))
        loss.backward()
        optimizer.step()

        log_metrics.append(dict(epoch=epoch, loss=loss.item(), metrics=metrics))
        # print(dict(epoch=epoch, loss=loss.item(), metrics=metrics))
        print_logs = "epoch no : {epoch}, batch no : {batch_no}, total loss : {loss},  classifier :{classifier}, mask: {mask} ==================="
        print(print_logs.format(epoch=epoch, batch_no=i, loss=loss.item(),  classifier=metrics['loss_classifier'], mask=metrics['loss_mask']))
        if (i % log_freq) == 0:
            yield log_metrics
            log_metrics = list()

    yield log_metrics

In [9]:
collate_fn = lambda _: tuple(zip(*_)) # one-liner, no need to import
train_dataset = build_features.LBD_Dataset(dataset_train_location, T.Compose([T.ToTensor()]))
test_dataset = build_features.LBD_Dataset(dataset_test_location, T.Compose([T.ToTensor()]))

train_data_loader = torch.utils.data.DataLoader(
        train_dataset, batch_size=train_config['batch_size'], shuffle=True, num_workers=4,
        collate_fn=collate_fn)

test_data_loader = torch.utils.data.DataLoader(
        test_dataset, batch_size=test_config['batch_size'], shuffle=False, num_workers=4,
        collate_fn=collate_fn)

In [10]:
import wandb
wandb_config = dict(
    project='LBD',
    entity='monika-ahirwar',
    config=dict(
        train_config=train_config,
        model_config=model_config,
        optim_config=optim_config,
    ),
    save_code=False,
    group='runs',
    job_type='train',
)

run = wandb.init(**wandb_config)
assert run is wandb.run # run was successfully initialized, is not None
run_id, run_dir = run.id, run.dir
exp_name = run.name
artifact_name = f'{run_id}-logs'

Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Currently logged in as: monika-ahirwar. Use `wandb login --relogin` to force relogin


In [11]:
from utils.engine import evaluate
# Train Data
for epoch in range(train_config['epochs']):
    # print(f'Epoch {epoch}=======================================>.')

    for logs in train_one_epoch(model, loss_fn, optimizer, train_data_loader, device, epoch=epoch, log_freq=1):
        for log in logs: 
            run.log(log)

    if epoch + 1 == train_config['epochs'] or epoch % train_config['ckpt_freq'] == 0:

        artifact = wandb.Artifact(artifact_name, type='files')
        with artifact.new_file(f'ckpt/{epoch}.pt', 'wb') as f:
            torch.save(model.state_dict(), f)
        run.log_artifact(artifact)

    if epoch % train_config['eval_freq'] == 0:
        eval_res = evaluate(run, model, test_data_loader, device=device)
    
    model.train(True)

epoch no : 0, batch no : 0, total loss : 3.7140488624572754,  classifier :1.3236035108566284, mask: 1.618571400642395 ===================
epoch no : 0, batch no : 1, total loss : 3.818882465362549,  classifier :1.0492355823516846, mask: 2.062462568283081 ===================
epoch no : 0, batch no : 2, total loss : 3.1449904441833496,  classifier :0.8328695297241211, mask: 1.616510033607483 ===================
epoch no : 0, batch no : 3, total loss : 2.14128041267395,  classifier :0.521739661693573, mask: 0.9232879281044006 ===================
epoch no : 0, batch no : 4, total loss : 1.918677568435669,  classifier :0.3011886775493622, mask: 0.9318251013755798 ===================
epoch no : 0, batch no : 5, total loss : 1.9294934272766113,  classifier :0.39691102504730225, mask: 0.8466576933860779 ===================
epoch no : 0, batch no : 6, total loss : 2.256957769393921,  classifier :0.18148396909236908, mask: 1.3628697395324707 ===================
epoch no : 0, batch no : 7, total 

In [12]:
dataset_base_dir = '/home/mahirwar/Desktop/Monika/npsad_data/monika/LBD/'
model_save_name = dataset_base_dir + "models/mrcnn_models/{name}_mrcnn_model_{epoch}.pth"
torch.save(model.state_dict(), model_save_name.format(name=exp_name, epoch=train_config['epochs']))


# print("\n =================The Model is Trained!====================")
# print("-----------------Visualizing Model predictions----------------")

# # TODO Testing is done on Individual WSI Folders
# input_path = '/mnt/new-nas/work/data/npsad_data/vivek/Datasets/amyb_wsi/test'

# model = build_default(model_config, im_size=1024)

# explain = ExplainPredictions(model, model_input_path = model_save_name.format(name=exp_name, epoch=train_config['epochs']), test_input_path=input_path, 
#                             detection_threshold=0.75, wandb=run, save_result=True, ablation_cam=True, save_thresholds=False)
# explain.generate_results()

run.finish()

epoch,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇███
loss,█▅▄▄▄▄▄▃▄▃▃▃▂▂▂▃▂▂▂▂▂▂▂▂▂▂▃▂▂▂▂▁▁▁▁▁▁▁▁▁
epoch,49
loss,0.3449
